<img src="logos/Logo_de_la_Facultad_Experimental_de_Ciencia_y_Tecnología.svg.png" width="250" style="display: block; margin-left: auto; margin-right: auto;">

# Titanic – Preparación de Datos e Ingeniería de Características

## Propósito

Este notebook convierte los principales hallazgos del EDA en un flujo de preprocesamiento claro y reproducible. La prioridad es comprender **qué hace cada paso y por qué es necesario**.

Al finalizar podremos:

- validar el contrato mínimo de los datos de entrada;
- crear particiones de train y test sin ajustar previamente ninguna transformación;
- crear un conjunto pequeño de características justificadas por el EDA;
- imputar, codificar y escalar diferentes tipos de variables;
- reunir todas las transformaciones en un pipeline de scikit-learn;
- preparar la entrega al notebook de comparación de modelos sin consultar el conjunto de test.

```text
Datos crudos
    ↓
Train / test
    ↓
Ingeniería de características
    ↓
Imputación + codificación + escalado
    ↓
Modelo
```

> Un notebook complementario avanzado contiene transformers personalizados, imputación condicional de edad, características basadas en grupos de boletos, pipelines configurables y experimentos de ablación. El presente notebook enseña primero el camino esencial.

> **Limitación del conjunto de test:** el EDA anterior examinó el conjunto completo de datos etiquetados. Por tanto, el conjunto de test es útil para el flujo de la asignatura, pero no constituye un test final perfectamente intacto. En un proyecto nuevo, el conjunto de test final debe crearse y bloquearse antes del EDA detallado.

## 1. Importaciones y rutas del proyecto

Las dependencias deben instalarse desde el entorno del proyecto, no desde una celda con `pip install` dentro del notebook.

In [1]:
from __future__ import annotations

import json
import platform
import re
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

RANDOM_STATE = 42
TEST_SIZE = 0.20
TARGET_COLUMN = "Survived"

# Permitir la ejecución desde la raíz del proyecto o desde notebooks/.
PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "data" / "titanic_data.csv").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "titanic_data.csv"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"

if not DATA_PATH.exists():
    raise FileNotFoundError(f"No se encontró el dataset en {DATA_PATH}")

print(f"Python: {platform.python_version()}")
print(f"pandas: {pd.__version__}")
print(f"scikit-learn: {sklearn.__version__}")

Python: 3.12.3
pandas: 3.0.1
scikit-learn: 1.8.0


## 2. Cargar y validar los datos crudos

El EDA identificó las columnas esperadas y los rangos válidos. Ahora convertimos esas observaciones en verificaciones ejecutables.

La validación se escribe deliberadamente mediante pasos explícitos. Si cambia el esquema de entrada, el notebook debe detenerse con un mensaje informativo en lugar de ignorar silenciosamente el problema.

In [2]:
required_columns = [
    "PassengerId",
    "Survived",
    "Pclass",
    "Name",
    "Sex",
    "Age",
    "SibSp",
    "Parch",
    "Ticket",
    "Fare",
    "Cabin",
    "Embarked",
]

df = pd.read_csv(DATA_PATH)

missing_columns = []
for column in required_columns:
    if column not in df.columns:
        missing_columns.append(column)

if missing_columns:
    raise ValueError(f"Faltan columnas requeridas: {missing_columns}")

if df["PassengerId"].duplicated().any():
    raise ValueError("PassengerId debe ser único.")

if not df["Pclass"].isin([1, 2, 3]).all():
    raise ValueError("Pclass solo puede contener 1, 2 o 3.")

if df["Age"].dropna().lt(0).any():
    raise ValueError("Age no puede ser negativa.")

if df["Fare"].dropna().lt(0).any():
    raise ValueError("Fare no puede ser negativa.")

if not df[TARGET_COLUMN].isin([0, 1]).all():
    raise ValueError("Survived solo puede contener 0 o 1.")

print(f"Dataset validado: {df.shape[0]} filas y {df.shape[1]} columnas")
df.head()

Dataset validado: 891 filas y 12 columnas


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## 3. Separar la variable objetivo y crear las particiones

Realizamos la partición antes de ajustar imputadores, codificadores, escaladores o modelos. `stratify=y` conserva aproximadamente la misma proporción de la variable objetivo en ambas particiones.

Solo los datos de entrenamiento pueden utilizarse para validación cruzada, selección de características, comparación de modelos, Optuna o selección del umbral. Deliberadamente no mostramos la distribución de la variable objetivo del conjunto de test.

In [3]:
X = df.drop(columns=[TARGET_COLUMN])
y = df[TARGET_COLUMN].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"Filas de entrenamiento: {len(X_train)}")
print(f"Filas de test: {len(X_test)}")

training_target_summary = (
    y_train
    .value_counts(normalize=True)
    .sort_index()
    .rename("proportion")
    .to_frame()
)
training_target_summary

Filas de entrenamiento: 712
Filas de test: 179


,proportion
Survived,
0,0.616573
1,0.383427


## 4. Traducir los hallazgos del EDA en decisiones de preparación

Este notebook implementa una única línea base fácil de comprender. Las representaciones alternativas permanecen como hipótesis para el notebook de modelado.

| Hallazgo del EDA | Decisión en esta línea base | Alternativa para comparar después |
|---|---|---|
| `Age` presenta ausencia estructurada | Imputación por mediana + indicador de ausencia | Imputación condicional por clase/título |
| `Fare` está fuertemente sesgada a la derecha | Usar `log1p(Fare)` | Conservar la tarifa original |
| `Pclass` puede no tener un efecto lineal | Codificación one-hot | Tratarla como ordinal para árboles |
| La estructura familiar es no lineal | Usar tamaño familiar e indicador de viaje en solitario | Conservar `SibSp` y `Parch` |
| La disponibilidad de cabina puede aportar información | Usar un indicador de cabina disponible | Usar la cubierta agrupada |
| El nombre contiene información social | Extraer y agrupar el título del pasajero | Excluir el título |

La línea base evita conservar simultáneamente representaciones exactamente redundantes.

## 5. Crear características comprensibles

La función siguiente no tiene **estado**: no aprende medianas, frecuencias de categorías ni información procedente de otras filas. Solamente combina valores ya presentes en el registro de cada pasajero.

Primero la presentamos como una función normal de pandas. Después, `FunctionTransformer` colocará esta misma función dentro del pipeline de sklearn.

In [4]:
def create_titanic_features(data: pd.DataFrame) -> pd.DataFrame:
    """Crear características deterministas a partir de las columnas crudas de Titanic."""
    result = data.copy()

    # Extraer el texto situado entre la coma y el punto del nombre del pasajero.
    title_pattern = r",\s*([^.]+)\."
    result["Title"] = result["Name"].str.extract(
        title_pattern,
        expand=False,
    )

    # Normalizar títulos equivalentes; el encoder agrupará los títulos raros.
    title_aliases = {
        "Mlle": "Miss",
        "Ms": "Miss",
        "Mme": "Mrs",
    }
    result["Title"] = result["Title"].replace(title_aliases)
    result["Title"] = result["Title"].fillna("Unknown")

    result["FamilySize"] = result["SibSp"] + result["Parch"] + 1
    result["IsAlone"] = (result["FamilySize"] == 1).astype(int)
    result["HasCabin"] = result["Cabin"].notna().astype(int)
    result["AgeMissing"] = result["Age"].isna().astype(int)
    result["FareLog1p"] = np.log1p(result["Fare"])

    return result

### Inspeccionar el resultado antes de construir el pipeline

Durante el desarrollo conviene inspeccionar algunas filas para verificar que cada característica nueva representa lo que pretendíamos. Esta vista previa tiene fines docentes y de depuración; la transformación final continuará ocurriendo dentro del pipeline.

In [5]:
preview = create_titanic_features(X_train.head())

columns_to_preview = [
    "SibSp",
    "Parch",
    "FamilySize",
    "IsAlone",
    "HasCabin",
    "Age",
    "AgeMissing",
    "Fare",
    "FareLog1p",
    "Title",
]

preview[columns_to_preview]

,SibSp,Parch,FamilySize,IsAlone,HasCabin,Age,AgeMissing,Fare,FareLog1p,Title
692,0,0,1,1,0,NaN,1,56.4958,4.051712,Mr
481,0,0,1,1,0,NaN,1,0.0000,0.000000,Mr
527,0,0,1,1,1,NaN,1,221.7792,5.406181,Mr
855,0,1,2,0,0,18.0,0,9.3500,2.336987,Mrs
801,1,1,3,0,0,31.0,0,26.2500,3.305054,Mrs


## 6. Agrupar las características según la transformación que necesitan

Un `ColumnTransformer` permite aplicar una estrategia de preparación distinta a cada tipo de variable.

| Grupo | Preparación |
|---|---|
| Numéricas | Imputación por mediana y estandarización |
| Categóricas | Imputación por moda y codificación one-hot |
| Indicadores binarios | Pasar sin modificaciones |

Variables como `PassengerId`, `Name`, `Cabin` y `Ticket` crudos, además de `SibSp` y `Parch`, no aparecen en estas listas. Por ello, `ColumnTransformer` las eliminará.

In [6]:
numeric_features = [
    "Age",
    "FareLog1p",
    "FamilySize",
]

categorical_features = [
    "Sex",
    "Embarked",
    "Pclass",
    "Title",
]

binary_features = [
    "AgeMissing",
    "IsAlone",
    "HasCabin",
]

## 7. Construir los pipelines numérico y categórico

### Variables numéricas

1. `SimpleImputer(strategy="median")` aprende una mediana con los datos de entrenamiento y completa los valores ausentes.
2. `StandardScaler()` centra y escala las variables numéricas. Esto resulta útil para numerosos modelos lineales y basados en distancias.

### Variables categóricas

1. Las categorías ausentes se sustituyen por la categoría de entrenamiento más frecuente.
2. La codificación one-hot crea columnas indicadoras.
3. Las categorías infrecuentes se agrupan utilizando únicamente los datos recibidos durante `fit`.
4. Una categoría que no se observó durante el entrenamiento puede procesarse de forma segura durante la inferencia.

In [7]:
numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                min_frequency=0.01,
                drop="if_binary",
            ),
        ),
    ]
)

## 8. Combinar las transformaciones con `ColumnTransformer`

`remainder="drop"` significa que solamente las características enumeradas explícitamente llegarán al modelo. Esto hace visible y auditable la selección de variables.

In [8]:
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features),
        ("binary", "passthrough", binary_features),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

## 9. Colocar la ingeniería de características dentro del pipeline de preparación

Aunque anteriormente inspeccionamos las características creadas, el flujo real debe volver a generarlas dentro del pipeline. Esto garantiza que el mismo código se aplique durante entrenamiento, validación, evaluación en test e inferencia futura.

El `fit_transform` siguiente se utiliza únicamente como demostración estructural sobre los datos de entrenamiento. Durante la comparación de modelos, el pipeline completo debe ajustarse por separado dentro de cada fold de validación cruzada.

In [9]:
feature_engineering = FunctionTransformer(
    create_titanic_features,
    validate=False,
)

preparation_pipeline = Pipeline(
    steps=[
        ("feature_engineering", feature_engineering),
        ("preprocessor", preprocessor),
    ]
)

X_train_prepared = preparation_pipeline.fit_transform(
    X_train,
    y_train,
)

output_feature_names = (
    preparation_pipeline
    .named_steps["preprocessor"]
    .get_feature_names_out()
)

print(f"Dimensiones crudas de entrenamiento: {X_train.shape}")
print(f"Dimensiones preparadas de entrenamiento: {X_train_prepared.shape}")

pd.DataFrame({"caracteristica_salida": output_feature_names})

Dimensiones crudas de entrenamiento: (712, 11)
Dimensiones preparadas de entrenamiento: (712, 18)


,caracteristica_salida
0,Age
1,FareLog1p
2,FamilySize
3,Sex_male
4,Embarked_C
5,Embarked_Q
6,Embarked_S
7,Pclass_1
8,Pclass_2
9,Pclass_3


### Verificaciones estructurales básicas

Estas verificaciones comprueban que:

- no cambió el número de pasajeros;
- todos los valores de salida son finitos.

In [10]:
assert X_train_prepared.shape[0] == X_train.shape[0]
assert X_train_prepared.shape[1] == len(output_feature_names)

if hasattr(X_train_prepared, "toarray"):
    values_to_check = X_train_prepared.toarray()
else:
    values_to_check = X_train_prepared

assert np.isfinite(values_to_check).all()

print("Todas las verificaciones estructurales fueron superadas.")

Todas las verificaciones estructurales fueron superadas.


## 10. Construir el futuro pipeline desde los datos crudos hasta el modelo

El notebook de modelado proporcionará un estimador. `clone` crea una copia no ajustada del preprocesador, de modo que los experimentos diferentes no compartan estado aprendido.

Por tanto, el modelo recibirá filas crudas de Titanic; la ingeniería de características, imputación, codificación, escalado y predicción permanecerán juntas.

In [11]:
def build_model_pipeline(estimator) -> Pipeline:
    """Crear un pipeline end-to-end no ajustado para un estimador determinado."""
    return Pipeline(
        steps=[
            (
                "feature_engineering",
                FunctionTransformer(
                    create_titanic_features,
                    validate=False,
                ),
            ),
            ("preprocessor", clone(preprocessor)),
            ("model", estimator),
        ]
    )

# Ejemplo para el notebook de modelado:
# from sklearn.linear_model import LogisticRegression
# model_pipeline = build_model_pipeline(
#     LogisticRegression(max_iter=2_000, random_state=RANDOM_STATE)
# )

## 11. Guardar un manifiesto legible de la partición

Guardamos la asignación de cada pasajero a una partición para que otro notebook pueda reproducir exactamente el split. La construcción es intencionalmente explícita: crear cada tabla, asignarle su etiqueta, combinar ambas tablas y ordenar el resultado.

**No** guardamos la matriz ya transformada. El pipeline final del modelo debe aprender su preprocesamiento dentro de la validación cruzada.

In [12]:
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

train_manifest = X_train[["PassengerId"]].copy()
train_manifest["split"] = "train"

test_manifest = X_test[["PassengerId"]].copy()
test_manifest["split"] = "test"

split_manifest = pd.concat(
    [train_manifest, test_manifest],
    ignore_index=True,
)

split_manifest = split_manifest.sort_values("PassengerId")

split_manifest.to_csv(
    ARTIFACTS_DIR / "split_assignments_teaching_es.csv",
    index=False,
)

split_manifest.head()

,PassengerId,split
607,1,train
209,2,train
142,3,train
633,4,train
810,5,test


## 12. Guardar metadatos mínimos de reproducibilidad

En esta etapa todavía no existe un modelo ajustado y seleccionado. Por ello no serializamos un pipeline de producción. Solo guardamos la información necesaria para reproducir la partición y el entorno.

In [13]:
metadata = {
    "random_state": RANDOM_STATE,
    "test_size": TEST_SIZE,
    "target_column": TARGET_COLUMN,
    "python_version": platform.python_version(),
    "pandas_version": pd.__version__,
    "scikit_learn_version": sklearn.__version__,
    "test_note": (
        "El EDA anterior examinó el conjunto completo de datos etiquetados. "
        "Debe tratarse como conjunto de test, no como un test final perfectamente intacto."
    ),
}

metadata_path = ARTIFACTS_DIR / "data_preparation_teaching_metadata_es.json"

with metadata_path.open("w", encoding="utf-8") as metadata_file:
    json.dump(metadata, metadata_file, indent=2)

print(f"Metadatos guardados en: {metadata_path}")

Metadatos guardados en: /mnt/c/Users/aespinoza/PycharmProjects/titanic_ml_solution/artifacts/data_preparation_teaching_metadata_es.json


## 13. Entrega al notebook de modelado

El notebook de modelado debe:

1. utilizar únicamente `X_train` y `y_train` para validación cruzada y selección de modelos;
2. comparar pipelines completos creados con `build_model_pipeline`;
3. comparar las decisiones de la línea base con las alternativas obtenidas del EDA;
4. seleccionar las métricas antes de inspeccionar los resultados de los modelos;
5. evaluar el conjunto de test únicamente después de seleccionar el pipeline completo.

### Experimentos sugeridos

- `Fare` original frente a `FareLog1p`;
- `Pclass` one-hot frente a ordinal;
- `SibSp`/`Parch` originales frente a `FamilySize`/`IsAlone`;
- excluir información de cabina frente a `HasCabin` frente a cubierta agrupada;
- imputación de edad por mediana frente a imputación condicional;
- excluir el título frente a utilizar el título agrupado;
- agregar una interacción explícita `Sex × Pclass` para modelos lineales;
- excluir el boleto frente a una característica de tamaño del grupo segura por fold.

El notebook complementario avanzado demuestra cómo implementar estas alternativas mediante transformers reutilizables y configurables.

### Comprueba tu comprensión

- ¿Por qué el imputador debe aprender la mediana únicamente con los datos de entrenamiento?
- ¿Por qué se excluye `PassengerId` del modelo?
- ¿Por qué no conservamos simultáneamente las variables familiares crudas y las derivadas en la línea base?
- ¿De qué nos protege `handle_unknown` durante la inferencia?
- ¿Por qué no debe utilizarse el conjunto de test durante la optimización con Optuna?